# 🏆 Preditor de Bola de Ouro — Ingestão Bronze
## Arquitetura Medallion no GCP | BigQuery ML

**Fluxo deste notebook:**
```
CSVs já no GCS ──► BigQuery (datasets Bronze / Silver / Gold)
```

### Pré-requisito: fazer upload manual dos CSVs no GCS

Antes de rodar este notebook, suba os arquivos abaixo para o bucket GCS via **Console GCP** ou `gsutil`:

| Arquivo local | Destino no GCS | Fonte |
|---|---|---|
| `players_stats_2017_2025.csv` | `bronze/players_stats_2017_2025.csv` | [Kaggle — emrey3lmaz](https://www.kaggle.com/datasets/emrey3lmaz/top-5-league-football-player-stats-2017-2025) |
| `players_stats_2122.csv` | `bronze/players_stats_2122.csv` | [Kaggle — vivovinco 21-22](https://www.kaggle.com/datasets/vivovinco/20212022-football-player-stats) |
| `players_stats_2223.csv` | `bronze/players_stats_2223.csv` | [Kaggle — vivovinco 22-23](https://www.kaggle.com/datasets/vivovinco/20222023-football-player-stats) |
| `ballon_dor_history.csv` | `bronze/ballon_dor_history.csv` | [Kaggle — mexwell (1956–2018)](https://www.kaggle.com/datasets/mexwell/ballon-dor-voting-history) |
| `ballon_dor_2019_2024.csv` | `bronze/ballon_dor_2019_2024.csv` | Criado manualmente pelo grupo |

**Como subir via Console GCP:**
1. Acesse [console.cloud.google.com/storage](https://console.cloud.google.com/storage)
2. Clique no seu bucket → pasta `bronze/`
3. Clique em **Upload Files** e selecione os CSVs

**Como subir via terminal (`gsutil`):**
```bash
gsutil cp players_stats_2017_2025.csv  gs://ballon-dor-raw-data/bronze/
gsutil cp players_stats_2122.csv       gs://ballon-dor-raw-data/bronze/
gsutil cp players_stats_2223.csv       gs://ballon-dor-raw-data/bronze/
gsutil cp ballon_dor_history.csv       gs://ballon-dor-raw-data/bronze/
gsutil cp ballon_dor_2019_2024.csv     gs://ballon-dor-raw-data/bronze/
```

---
## ⚙️ Configurações — edite só esta célula

In [ ]:
# --- projeto GCP detectado automaticamente (não precisa editar ao trocar de projeto) ---
import os, subprocess

def get_project_id():
    for var in ("GOOGLE_CLOUD_PROJECT", "GCLOUD_PROJECT", "CLOUDSDK_CORE_PROJECT"):
        if os.environ.get(var):
            return os.environ[var]
    try:
        import google.auth
        _, proj = google.auth.default()
        if proj:
            return proj
    except Exception:
        pass
    try:
        proj = subprocess.run(["gcloud", "config", "get-value", "project"],
                              capture_output=True, text=True).stdout.strip()
        if proj and proj != "(unset)":
            return proj
    except Exception:
        pass
    raise RuntimeError('Projeto não encontrado. Rode: os.environ["GOOGLE_CLOUD_PROJECT"] = "seu-projeto"')

PROJECT_ID = get_project_id()
print("Projeto GCP:", PROJECT_ID)


In [1]:
BUCKET_NAME = "futdados"
LOCATION    = "US"

# {nome_tabela_bq: (uri_gcs, separador_csv)}
BRONZE_TABLES = {
    "bronze.players_stats_main_raw" : (f"gs://{BUCKET_NAME}/bronze/players_stats_2017_2025.csv", ";"), # emrey3lmaz usa ponto-e-vírgula
    "bronze.players_stats_2122_raw" : (f"gs://{BUCKET_NAME}/bronze/players_stats_2122.csv",      ";"), # vivovinco usa ponto-e-vírgula
    "bronze.players_stats_2223_raw" : (f"gs://{BUCKET_NAME}/bronze/players_stats_2223.csv",      ";"), # vivovinco usa ponto-e-vírgula
    "bronze.ballon_dor_history_raw" : (f"gs://{BUCKET_NAME}/bronze/ballon_dor_history.csv",      ","), # mexwell usa vírgula
    "bronze.ballon_dor_recent_raw"  : (f"gs://{BUCKET_NAME}/bronze/ballon_dor_2019_2024.csv",    ","), # manual usa vírgula
}

print("✅ Configurações carregadas!")
print(f"   Projeto  : {PROJECT_ID}")
print(f"   Bucket   : gs://{BUCKET_NAME}")
print(f"   Tabelas  : {len(BRONZE_TABLES)} tabelas Bronze a criar")

✅ Configurações carregadas!
   Projeto  : trabalho-pdm-joao-fut
   Bucket   : gs://futdados
   Tabelas  : 5 tabelas Bronze a criar


---
## 📦 Passo 1 — Instalar e Importar Dependências

In [3]:
!pip install google-cloud-storage google-cloud-bigquery pandas
print("✅ Dependências instaladas!")

✅ Dependências instaladas!


In [4]:
import pandas as pd
from google.cloud import storage, bigquery
from google.cloud.exceptions import NotFound

storage_client = storage.Client(project=PROJECT_ID)
bq_client      = bigquery.Client(project=PROJECT_ID)

print("✅ Clientes GCP inicializados!")

✅ Clientes GCP inicializados!


---
## ☁️ Passo 2 — Verificar os Arquivos no GCS

In [5]:
print(f"📂 Arquivos em gs://{BUCKET_NAME}/bronze/\n")

blobs = list(storage_client.list_blobs(BUCKET_NAME, prefix="bronze/"))

if not blobs:
    print("⚠️  Nenhum arquivo encontrado! Faça o upload dos CSVs antes de continuar.")
else:
    esperados = {
        "bronze/players_stats_2017_2025.csv",
        "bronze/players_stats_2122.csv",
        "bronze/players_stats_2223.csv",
        "bronze/ballon_dor_history.csv",
        "bronze/ballon_dor_2019_2024.csv",
    }
    presentes = set()

    for blob in blobs:
        size_kb = blob.size / 1024
        status  = "✅" if blob.name in esperados else "📄"
        print(f"  {status} {blob.name:<50} {size_kb:>8.1f} KB")
        presentes.add(blob.name)

    faltando = esperados - presentes
    if faltando:
        print("\n⚠️  Arquivos faltando no GCS:")
        for f in sorted(faltando):
            print(f"   ❌ {f}")
    else:
        print("\n✅ Todos os arquivos necessários estão no GCS!")

📂 Arquivos em gs://futdados/bronze/

  📄 bronze/                                                 0.0 KB
  ✅ bronze/ballon_dor_2019_2024.csv                         6.6 KB
  ✅ bronze/ballon_dor_history.csv                         146.0 KB
  ✅ bronze/players_stats_2017_2025.csv                  12748.6 KB
  ✅ bronze/players_stats_2122.csv                        1680.8 KB
  ✅ bronze/players_stats_2223.csv                        1301.1 KB

✅ Todos os arquivos necessários estão no GCS!


---
## 🗄️ Passo 3 — Criar os Datasets BigQuery (Bronze / Silver / Gold)

In [6]:
DATASETS = {
    "bronze": "🥉 Dados brutos — cópia fiel das fontes originais",
    "silver": "🥈 Dados limpos, filtrados e enriquecidos",
    "gold"  : "🥇 Dados agregados e prontos para ML e BI",
}

print("🗄️  Criando datasets no BigQuery...\n")
for ds_id, desc in DATASETS.items():
    ds             = bigquery.Dataset(f"{PROJECT_ID}.{ds_id}")
    ds.location    = LOCATION
    ds.description = desc
    bq_client.create_dataset(ds, exists_ok=True)
    print(f"  ✅ {ds_id:8s} — {desc}")

print(f"\n🔗 https://console.cloud.google.com/bigquery?project={PROJECT_ID}")

🗄️  Criando datasets no BigQuery...

  ✅ bronze   — 🥉 Dados brutos — cópia fiel das fontes originais
  ✅ silver   — 🥈 Dados limpos, filtrados e enriquecidos
  ✅ gold     — 🥇 Dados agregados e prontos para ML e BI

🔗 https://console.cloud.google.com/bigquery?project=trabalho-pdm-joao-fut


---
## 📥 Passo 4 — Carregar Tabelas Bronze no BigQuery

> **Nota técnica:** Utilizamos `column_name_character_map="V2"` para permitir que o BigQuery lide graciosamente com nomes de colunas que tenham caracteres inválidos (como `G/Sh`), fazendo o fallback seguro caso a feature não esteja disponível na API.

In [7]:
def load_csv_to_bq(table_id: str, gcs_uri: str, sep: str = ",") -> None:
    full_id = f"{PROJECT_ID}.{table_id}"

    # Configuração base
    base_cfg_dict = dict(
        source_format         = bigquery.SourceFormat.CSV,
        field_delimiter       = sep,
        skip_leading_rows     = 1,
        autodetect            = True,
        write_disposition     = bigquery.WriteDisposition.WRITE_TRUNCATE,
        allow_quoted_newlines = True,
        allow_jagged_rows     = True,
    )

    filename = gcs_uri.split("/")[-1]
    print(f"  ⏳ {filename:<45} → {table_id}")

    # Tenta carregar com o mapeamento de caracteres V2 (para colunas como G/Sh)
    for use_v2 in [True, False]:
        cfg_dict = dict(base_cfg_dict)
        if use_v2:
            try:
                cfg_dict["column_name_character_map"] = "V2"
                cfg = bigquery.LoadJobConfig(**cfg_dict)
            except TypeError:
                continue  # V2 não suportado nesta versão da biblioteca
        else:
            cfg = bigquery.LoadJobConfig(**cfg_dict)

        try:
            job = bq_client.load_table_from_uri(gcs_uri, full_id, job_config=cfg)
            job.result()
            table = bq_client.get_table(full_id)
            print(f"     ✅ {table.num_rows:>7,} linhas | {len(table.schema):>3} colunas | sep='{sep}'\n")
            return
        except Exception as e:
            if use_v2:
                print(f"     ⚠️  Falha com character map V2, tentando sem...")
            else:
                print(f"     ❌ Erro em {table_id}: {e}\n")
                raise

print("📥 Carregando tabelas Bronze no BigQuery...\n")

erros = []
for table_id, (gcs_uri, sep) in BRONZE_TABLES.items():
    try:
        load_csv_to_bq(table_id, gcs_uri, sep)
    except Exception:
        erros.append(table_id)

if erros:
    print(f"\n⚠️  {len(erros)} tabela(s) com erro: {erros}")
else:
    print("🎉 Camada Bronze carregada com sucesso!")

📥 Carregando tabelas Bronze no BigQuery...

  ⏳ players_stats_2017_2025.csv                   → bronze.players_stats_main_raw
     ✅  22,929 linhas | 178 colunas | sep=';'

  ⏳ players_stats_2122.csv                        → bronze.players_stats_2122_raw
     ✅   2,921 linhas | 143 colunas | sep=';'

  ⏳ players_stats_2223.csv                        → bronze.players_stats_2223_raw
     ✅   2,689 linhas | 124 colunas | sep=';'

  ⏳ ballon_dor_history.csv                        → bronze.ballon_dor_history_raw
     ✅   1,780 linhas |  16 colunas | sep=','

  ⏳ ballon_dor_2019_2024.csv                      → bronze.ballon_dor_recent_raw
     ✅     150 linhas |   6 colunas | sep=','

🎉 Camada Bronze carregada com sucesso!


---
## ✅ Passo 5 — Verificar as Tabelas Bronze

In [8]:
# Contagem segura de registros (ignora tabelas que falharam)
print("📊 Registros nas tabelas Bronze:\n")
resultados = []

for table_id in BRONZE_TABLES:
    full_id = f"{PROJECT_ID}.{table_id}"
    try:
        tbl = bq_client.get_table(full_id)
        resultados.append({
            "tabela"   : table_id,
            "linhas"   : f"{tbl.num_rows:,}",
            "colunas"  : len(tbl.schema),
            "status"   : "✅ OK"
        })
    except NotFound:
        resultados.append({
            "tabela"   : table_id,
            "linhas"   : 0,
            "colunas"  : 0,
            "status"   : "❌ Não encontrada"
        })

df_counts = pd.DataFrame(resultados)
df_counts

📊 Registros nas tabelas Bronze:



,tabela,linhas,colunas,status
0,bronze.players_stats_main_raw,"22,929",178,✅ OK
1,bronze.players_stats_2122_raw,"2,921",143,✅ OK
2,bronze.players_stats_2223_raw,"2,689",124,✅ OK
3,bronze.ballon_dor_history_raw,"1,780",16,✅ OK
4,bronze.ballon_dor_recent_raw,150,6,✅ OK


In [10]:
# Amostra do dataset principal de jogadores (emrey3lmaz)
try:
    q = f"SELECT * FROM `{PROJECT_ID}.bronze.players_stats_main_raw` LIMIT 5"
    print("🔍 Amostra — bronze.players_stats_main_raw:")
    display(bq_client.query(q).to_dataframe())
except NotFound:
    print("❌ Tabela bronze.players_stats_main_raw não encontrada.")

🔍 Amostra — bronze.players_stats_main_raw:


,league,season,team,player,nation_,pos_,age_,born_,Playing Time_MP,Playing Time_Starts,...,Performance_Crs,Performance_Fld,Performance_Fls,Performance_Int,Performance_OG,Performance_Off,Performance_PKcon,Performance_PKwon,Performance_Recov,Performance_TklW
0,FRA-Ligue 1,2425,Paris S-G,Randal Kolo Muani,FRA,FW,25,1998,10,2,...,0,7,7,0,0,2,0,1,8,4
1,ESP-La Liga,2223,Valencia,Diego López,ESP,"FW,MF",20,2002,10,5,...,12,7,6,3,0,1,0,0,29,2
2,ITA-Serie A,1718,Fiorentina,Khouma Babacar,SEN,FW,24,1993,16,1,...,5,12,9,2,0,8,0,1,18,4
3,ITA-Serie A,1718,Napoli,Arkadiusz Milik,POL,"FW,MF",23,1994,15,3,...,2,7,8,1,0,1,0,0,11,1
4,GER-Bundesliga,2425,Wolfsburg,Lukas Nmecha,GER,"FW,MF",25,1998,19,3,...,6,17,7,2,0,0,0,0,15,0


In [11]:
# Colunas do dataset principal (emrey3lmaz) — útil para escrever o SQL da Silver
try:
    table = bq_client.get_table(f"{PROJECT_ID}.bronze.players_stats_main_raw")
    print(f"Colunas em players_stats_main_raw ({len(table.schema)} colunas):")
    for i, field in enumerate(table.schema, 1):
        print(f"  {i:3}. {field.name:<45} {field.field_type}")
except NotFound:
    print("❌ Tabela bronze.players_stats_main_raw não encontrada.")

Colunas em players_stats_main_raw (178 colunas):
    1. league                                        STRING
    2. season                                        INTEGER
    3. team                                          STRING
    4. player                                        STRING
    5. nation_                                       STRING
    6. pos_                                          STRING
    7. age_                                          INTEGER
    8. born_                                         INTEGER
    9. Playing Time_MP                               INTEGER
   10. Playing Time_Starts                           INTEGER
   11. Playing Time_Min                              INTEGER
   12. Playing Time_90s                              INTEGER
   13. Performance_Gls                               INTEGER
   14. Performance_Ast                               INTEGER
   15. Performance_G+A                               INTEGER
   16. Performance_G-PK                  

In [ ]:
# Temporadas disponíveis no dataset principal
try:
    q = f"""
    SELECT season, COUNT(*) AS jogadores
    FROM `{PROJECT_ID}.bronze.players_stats_main_raw`
    GROUP BY season ORDER BY season DESC
    """
    print("⚽ Temporadas no players_stats_main_raw:")
    display(bq_client.query(q).to_dataframe())
except NotFound:
    print("❌ Tabela bronze.players_stats_main_raw não encontrada.")
except Exception as e:
    print(f"❌ Erro ao consultar temporadas: {e}")

In [ ]:
# Cobertura completa do Ballon d'Or (histórico + recente)
try:
    q = f"""
    SELECT year, COUNT(*) AS nomeados, MAX(points) AS pontos_vencedor
    FROM (
      SELECT SAFE_CAST(year AS INT64) AS year, SAFE_CAST(points AS FLOAT64) AS points
      FROM `{PROJECT_ID}.bronze.ballon_dor_history_raw`
      WHERE SAFE_CAST(year AS INT64) IS NOT NULL
      UNION ALL
      SELECT year, points FROM `{PROJECT_ID}.bronze.ballon_dor_recent_raw`
    )
    GROUP BY year ORDER BY year DESC LIMIT 20
    """
    print("🏆 Cobertura do Ballon d'Or (anos mais recentes):")
    display(bq_client.query(q).to_dataframe())
except NotFound:
    print("❌ Uma ou ambas as tabelas do Ballon d'Or não foram encontradas.")

---
## 🎯 Resumo da Ingestão Bronze

| Tabela BigQuery | Conteúdo | Sep |
|---|---|---|
| `bronze.players_stats_main_raw` | Stats jogadores 2017-2025 (emrey3lmaz) | `;` |
| `bronze.players_stats_2122_raw` | Stats jogadores 2021-22 (vivovinco) | `;` |
| `bronze.players_stats_2223_raw` | Stats jogadores 2022-23 (vivovinco) | `;` |
| `bronze.ballon_dor_history_raw` | Ballon d'Or histórico 1956-2018 (mexwell) | `,` |
| `bronze.ballon_dor_recent_raw`  | Ballon d'Or 2019-2024 (grupo) | `,` |

### ➡️ Próximos passos — BigQuery Console

```
1. sql/bronze/02_union_ballondor.sql        → view unificada Ballon d'Or (1956–2024)
2. sql/silver/01_create_silver_players.sql  → limpa jogadores, calcula métricas/90min
3. sql/silver/02_create_silver_ballondor.sql→ cria ballon_dor_score (target)
4. sql/silver/03_create_silver_joined.sql   → join jogadores × Ballon d'Or
5. sql/gold/01_create_gold_features.sql     → tabela Gold pronta para ML
6. sql/ml/01_create_model.sql               → treina o modelo (LINEAR_REG)
7. sql/ml/02_evaluate_model.sql             → métricas + feature importance
8. sql/ml/03_predict_ranking.sql            → Top-10 previsto vs. real (demo ao vivo)
```